# Evaluating RAG

Every notebook of this chapter added choices: chunk sizes, retrievers, rerankers, prompts, an
agent loop. This notebook follows the book section *Evaluating RAG* and measures the main
choice of the chapter, a fixed pipeline against an agent, on a small benchmark. We score the
retrieved context, the answers, and the price of each answer, and then ask the book's question:
is the difference real, or could it be chance?

**Learning goals:**
- Build a small RAG benchmark with expected facts per question
- Measure context recall and context precision of what each system retrieved
- Measure latency (p95), tokens, and tool calls: the price of an answer
- Score correctness, completeness, and faithfulness, and check an LLM judge against a rule
- Compare two systems with the paired sign test, and see when RAG is not needed at all

**Prerequisites:** ch07-02 (the pipeline), ch07-04 (the agent), Chapter 2 (evaluation, paired tests)

## Setup

In [1]:
import os
import json
import math
import re
import threading
import unicodedata
from concurrent.futures import ThreadPoolExecutor

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

import logging
import warnings
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

from shared.display import print_table, display_md
from shared.collections import load_collection
from ch07_rag import build_units, SparseIndex, DenseIndex, retrieve, build_prompt
from ch07_agent import Runtime, make_tools, run_agent, load_runs, save_runs, RUNS_FILE

<div style="border-left: 4px solid #64748B; background: rgba(100, 116, 139, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#64748B; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">AWS and recorded runs</strong><br>
The benchmark makes about 150 model calls: the three systems answer 24 questions, and a judge scores the answers. A live run takes a few minutes. Without AWS credentials, every call replays a run recorded earlier, with its answers, tool results, token counts, and timings, so all numbers below are reproducible offline.
</div>

The knowledge base and the retrieval chain are those of ch07-02. The retriever's models are
shared by all questions, so a lock lets several questions run in parallel without two of them
using the models at the same moment.

In [2]:
collection = load_collection("nobel-2025")
parents, children = build_units(collection, parent_size=2000, child_size=400)

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
sparse = SparseIndex(parents)
dense = DenseIndex(children, bi_encoder)
lock = threading.Lock()

def search(question, k=5):
    """Top-k parents for a question: hybrid retrieval, then cross-encoder reranking."""
    with lock:
        return [p for p, _ in retrieve(question, sparse, dense, children, parents,
                                       cross_encoder, top_k=k)]

tools = make_tools(search, parents)
EVAL_RUNS = RUNS_FILE.parent / "eval_runs.json"
recording = load_runs(EVAL_RUNS)

## The Benchmark

The book's harness starts from a fixed benchmark: questions, each with a reference answer and
the evidence it requires. We keep it simple and checkable: each question comes with a list of
**expected facts**, short strings such as a name or a university that a complete answer must
contain. Half of the questions need one lookup; the other half need a lookup per person, the
multi-hop shape of ch07-03 and ch07-04. Some ask about 2026, past the model's training data.

In [3]:
BENCHMARK = [
    # (id, kind, question, expected facts)
    ("s01", "single", "Who won the 2026 Nobel Prize in Physics?", ["Halzen"]),
    ("s02", "single", "Who won the 2025 Nobel Prize in Physics?", ["Clarke", "Devoret", "Martinis"]),
    ("s03", "single", "Who received the 2024 physics Nobel together with Geoffrey Hinton?", ["Hopfield"]),
    ("s04", "single", "Which chemists shared the 2025 Nobel Prize in Chemistry?", ["Kitagawa", "Robson", "Yaghi"]),
    ("s05", "single", "Who received the 2025 Nobel Prize in Physiology or Medicine?", ["Brunkow", "Ramsdell", "Sakaguchi"]),
    ("s06", "single", "Who was awarded the Nobel Peace Prize in 2025?", ["Machado"]),
    ("s07", "single", "Who won the 2026 Nobel Prize in Literature?", ["Anne Carson"]),
    ("s08", "single", "Who won the 2026 Nobel Peace Prize?", ["Pillay"]),
    ("s09", "single", "Who shared the 2026 Nobel Prize in Chemistry?", ["Kagan", "Soai"]),
    ("s10", "single", "Who received the 2026 Nobel Prize in Economic Sciences?", ["Hall", "Woodford"]),
    ("s11", "single", "What is a transmon?", ["charge qubit"]),
    ("s12", "single", "In which year did John Martinis found Qolab?", ["2022"]),
    ("m01", "multi", "Who won the 2025 Nobel Prize in Physics, and which universities are they affiliated with?",
     ["Clarke", "Devoret", "Martinis", "Berkeley", "Santa Barbara"]),
    ("m02", "multi", "Where do the 2025 Nobel laureates in physics work today?", ["Berkeley", "Santa Barbara"]),
    ("m03", "multi", "Which universities are the 2024 physics laureates affiliated with?", ["Princeton", "Toronto"]),
    ("m04", "multi", "At which university is the 2026 physics laureate a professor?", ["Wisconsin"]),
    ("m05", "multi", "Where did the 2026 physics laureate study?", ["Leuven"]),
    ("m06", "multi", "Where do the 2025 chemistry laureates work?", ["Kyoto", "Melbourne"]),
    ("m07", "multi", "Which universities are the 2025 economics laureates affiliated with?",
     ["Northwestern", "Collège de France", "Brown"]),
    ("m08", "multi", "Where do the 2025 laureates in physiology or medicine work?",
     ["Institute for Systems Biology", "Sonoma", "Osaka"]),
    ("m09", "multi", "Who was John Martinis's doctoral advisor, and at which university is that "
     "advisor a professor emeritus?", ["Clarke", "Berkeley"]),
    ("m10", "multi", "Which 2025 physics laureate is chief scientist for quantum hardware at a "
     "technology company, and which company is it?", ["Devoret", "Google"]),
    ("m11", "multi", "Which 2025 Nobel laureates, in any field, are affiliated with Kyoto University?",
     ["Kitagawa", "Sakaguchi"]),
    ("m12", "multi", "Which 2025 economics laureate holds a chair at the Collège de France, and "
     "who is his long-time co-author?", ["Aghion", "Howitt"]),
]
YEAR_2026 = {"s01", "s07", "s08", "s09", "s10", "m04", "m05"}

display_md(f"**{len(BENCHMARK)} questions**: {sum(k == 'single' for _, k, _, _ in BENCHMARK)} "
           f"single lookups, {sum(k == 'multi' for _, k, _, _ in BENCHMARK)} multi-hop; "
           f"{len(YEAR_2026)} about 2026.")

**24 questions**: 12 single lookups, 12 multi-hop; 7 about 2026.

Whether a text contains a fact is a rule, not a judgement: we compare whole words, ignoring
case and accents, so "College de France" matches "Collège de France" and "Hall" does not match
"Halliday".

In [4]:
def fold(text):
    """Lower case, accents removed: 'Collège' -> 'college'."""
    return "".join(c for c in unicodedata.normalize("NFKD", text)
                   if not unicodedata.combining(c)).lower()

def has(text, fact):
    return re.search(rf"\b{re.escape(fold(fact))}\b", fold(text)) is not None

## Three Systems

We compare the two systems of the chapter and add a third as a floor:

- **A, pipeline:** the fixed chain of ch07-02. Hybrid retrieval and reranking select 5
  passages, and the generator answers with the grounded prompt.
- **B, agent:** the loop of ch07-04 with its three tools.
- **C, model only:** the same model with no retrieval at all.

Each system returns its answer and its **context**: the pieces of evidence it put in front of
the generator. For the pipeline these are its 5 passages; for the agent, every passage, record,
and snippet its tools returned.

In [5]:
def evidence_items(trace):
    """The agent's context: each passage, record, or web snippet its tools returned."""
    items = []
    for t in trace:
        result = json.loads(t["result"])
        if isinstance(result, dict) and "error" in result:
            continue                                   # a miss is not evidence
        for r in result if isinstance(result, list) else [result]:
            items.append(r.get("text") or r.get("snippet") or json.dumps(r, ensure_ascii=False))
    return items

def system_a(qid, question):
    rt = Runtime(f"A-{qid}", tools, store=recording)
    ids = search(question, 5)
    prompt = build_prompt(question, ids, parents)
    answer = rt.llm([{"role": "user", "content": prompt}], []).text
    rt.save()
    return dict(answer=answer, context=[parents[p]["body"] for p in ids], rt=rt)

def system_b(qid, question):
    rt = Runtime(f"B-{qid}", tools, store=recording)
    answer = run_agent(question, rt)
    rt.save()
    return dict(answer=answer, context=evidence_items(rt.trace), rt=rt)

def system_c(qid, question):
    rt = Runtime(f"C-{qid}", tools, store=recording)
    answer = rt.llm([{"role": "user", "content": question + " Answer briefly."}], []).text
    rt.save()
    return dict(answer=answer, context=[], rt=rt)

SYSTEMS = {"A pipeline": system_a, "B agent": system_b, "C model only": system_c}

results = {}
for name, fn in SYSTEMS.items():
    with ThreadPoolExecutor(max_workers=4) as pool:
        runs = pool.map(lambda item: fn(item[0], item[2]), BENCHMARK)
        results[name] = dict(zip([qid for qid, *_ in BENCHMARK], runs))

save_runs(recording, EVAL_RUNS)
live = all(r["rt"].live for rs in results.values() for r in rs.values())
display_md(f"**{sum(len(r) for r in results.values())} answers** "
           f"{'computed live' if live else 'replayed from the recording'}.")

**72 answers** computed live.

## Retrieval Still Sets the Ceiling

**Context recall** asks whether the context holds every expected fact; **context precision**
asks which share of the context items hold at least one. Both follow from the facts by rule.
The model-only system has no context, so it gets no score.

In [6]:
def context_recall(context, facts):
    return sum(any(has(c, f) for c in context) for f in facts) / len(facts)

def context_precision(context, facts):
    return sum(any(has(c, f) for f in facts) for c in context) / len(context) if context else 0.0

for name in ["A pipeline", "B agent"]:
    for qid, kind, q, facts in BENCHMARK:
        r = results[name][qid]
        r["recall"] = context_recall(r["context"], facts)
        r["precision"] = context_precision(r["context"], facts)

def mean(name, key, kind=None):
    vals = [results[name][qid][key] for qid, k, *_ in BENCHMARK if kind in (None, k)]
    return sum(vals) / len(vals)

print_table([[name, f"{mean(name, 'recall', 'single'):.2f}", f"{mean(name, 'recall', 'multi'):.2f}",
              f"{mean(name, 'precision'):.2f}",
              f"{np.mean([len(results[name][q]['context']) for q, *_ in BENCHMARK]):.1f}"]
             for name in ["A pipeline", "B agent"]],
            ["System", "Context recall (single)", "Context recall (multi)", "Context precision",
             "Context items"])

| System     |   Context recall (single) |   Context recall (multi) |   Context precision |   Context items |
|:-----------|--------------------------:|-------------------------:|--------------------:|----------------:|
| A pipeline |                      0.83 |                     0.52 |                0.36 |             5   |
| B agent    |                      1    |                     1    |                0.47 |             7.1 |

The book's worked example, live: context recall and precision for the running question.

In [7]:
qid, _, q, facts = BENCHMARK[12]
rows = []
for name in ["A pipeline", "B agent"]:
    r = results[name][qid]
    found = [f for f in facts if any(has(c, f) for c in r["context"])]
    rows.append([name, len(r["context"]), ", ".join(found),
                 ", ".join(f for f in facts if f not in found) or "-",
                 f"{r['recall']:.2f}", f"{r['precision']:.2f}"])
display_md(f"**{q}**")
print_table(rows, ["System", "Items", "Facts in context", "Missing", "Recall", "Precision"])

**Who won the 2025 Nobel Prize in Physics, and which universities are they affiliated with?**

| System     |   Items | Facts in context                                   | Missing                 |   Recall |   Precision |
|:-----------|--------:|:---------------------------------------------------|:------------------------|---------:|------------:|
| A pipeline |       5 | Clarke, Devoret, Martinis                          | Berkeley, Santa Barbara |      0.6 |        0.4  |
| B agent    |       8 | Clarke, Devoret, Martinis, Berkeley, Santa Barbara | -                       |      1   |        0.88 |

The pipeline's five passages already miss facts on a few single lookups, and on multi-hop
questions they hold only about half of what an answer needs. No generator can recover a fact
that never reached it. The agent's follow-up lookups bring every expected fact in, and because
each lookup targets one person or one prize, its context is also more precise than five passages
retrieved for the whole question.

The running question shows the book's worked example with real numbers. The pipeline finds the
laureates but none of their universities. The agent finds everything; its precision stays below
1 because each search returns three passages, and not all of them are on target.

## The Price of an Answer

Every system call is timed and every token counted. The book asks for latency as the 95th
percentile, the slow cases users notice, and for cost per answered question.

In [8]:
def p95(values):
    return float(np.percentile(values, 95))

rows = []
for name in SYSTEMS:
    rs = [results[name][qid] for qid, *_ in BENCHMARK]
    rows.append([name, f"{np.median([r['rt'].seconds for r in rs]):.1f}",
                 f"{p95([r['rt'].seconds for r in rs]):.1f}",
                 f"{np.mean([r['rt'].tokens for r in rs]):,.0f}",
                 f"{np.mean([r['rt'].calls for r in rs]):.1f}",
                 f"{np.mean([len(r['rt'].trace) for r in rs]):.1f}"])
print_table(rows, ["System", "Median s", "p95 s", "Tokens per answer", "Model calls",
                   "Tool calls"])

| System       |   Median s |   p95 s |   Tokens per answer |   Model calls |   Tool calls |
|:-------------|-----------:|--------:|--------------------:|--------------:|-------------:|
| A pipeline   |        1.8 |     3.2 |               1,930 |           1   |          0   |
| B agent      |        8.3 |    16.1 |               5,958 |           2.7 |          3.4 |
| C model only |        2   |     4.3 |                 258 |           1   |          0   |

The seconds count model and tool calls, not the retrieval of the pipeline, which takes well under
a second. Four questions ran in parallel, so the absolute times include some queueing; the
ratios between systems are what matter. The agent takes several times as long and uses about
three times the tokens of the pipeline, because every step resends the whole conversation, tool
results included. Its p95 shows the slow cases: the questions where it needed several rounds.

## Measuring Answer Quality

Two of the book's four dimensions follow from the expected facts by rule:

- **Correctness:** the answer contains every expected fact.
- **Completeness:** the share of expected facts the answer contains.

In [9]:
for name in SYSTEMS:
    for qid, kind, q, facts in BENCHMARK:
        r = results[name][qid]
        r["completeness"] = sum(has(r["answer"], f) for f in facts) / len(facts)
        r["correct"] = r["completeness"] == 1.0

**Faithfulness** needs a judge. As in the book, the judge splits the answer into claims and
checks each one against the system's context; faithfulness is the share of supported claims.
The same call also judges correctness against the expected facts, which we use in a moment to
check the judge itself. The judge answers in plain lines, one per claim, which is easier to
parse reliably than JSON with quotations inside.

In [10]:
JUDGE = """You evaluate an answer of a question-answering system.

<question>{question}</question>
<expected_facts>{facts}</expected_facts>
<context>
{context}
</context>
<answer>{answer}</answer>

1. Split the answer into its factual claims. For each claim, decide whether the context
   supports it. Statements that the context lacks information are not claims.
2. Decide whether the answer is correct: it states every expected fact as the answer to the
   question.

Reply with one line per claim, starting with SUPPORTED: or UNSUPPORTED:, followed by the
claim. End with one line CORRECT: yes or CORRECT: no. Write nothing else."""

def judge(name, qid, question, facts):
    r = results[name][qid]
    rt = Runtime(f"judge-{name[0]}-{qid}", tools, store=recording)
    reply = rt.ask(JUDGE.format(question=question, facts=", ".join(facts),
                                context="\n\n".join(r["context"]), answer=r["answer"]),
                   max_tokens=3000)
    rt.save()
    claims = [{"claim": m.group(2).strip(), "supported": m.group(1) == "SUPPORTED"}
              for m in re.finditer(r"^(SUPPORTED|UNSUPPORTED):(.*)$", reply, re.M)]
    r["claims"] = claims
    r["faithfulness"] = sum(c["supported"] for c in claims) / len(claims) if claims else 1.0
    r["judge_correct"] = re.search(r"^CORRECT:\s*yes", reply, re.M | re.I) is not None

with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(lambda job: judge(*job),
                  [(name, qid, q, facts) for name in ["A pipeline", "B agent"]
                   for qid, _, q, facts in BENCHMARK]))
save_runs(recording, EVAL_RUNS)

rows = []
for name in SYSTEMS:
    rs = [results[name][qid] for qid, *_ in BENCHMARK]
    rows.append([name, f"{sum(r['correct'] for r in rs)} / {len(rs)}",
                 f"{np.mean([r['completeness'] for r in rs]):.2f}",
                 f"{np.mean([r['faithfulness'] for r in rs]):.2f}" if "faithfulness" in rs[0] else "-"])
print_table(rows, ["System", "Correct", "Completeness", "Faithfulness"])

| System       | Correct   |   Completeness | Faithfulness   |
|:-------------|:----------|---------------:|:---------------|
| A pipeline   | 14 / 24   |           0.64 | 0.98           |
| B agent      | 23 / 24   |           0.96 | 0.97           |
| C model only | 16 / 24   |           0.67 | -              |

Faithful is not the same as correct, as the book warns. The pipeline is almost perfectly
faithful, yet gets far fewer answers right: when its passages lack a fact, its grounded prompt
makes it say so instead of guessing. The model alone has no context to be faithful to, and still
answers two thirds of the questions correctly from memory.

One judged answer in full shows what faithfulness counts:

In [11]:
r = results["A pipeline"]["m01"]
display_md(f"**A pipeline, m01:** {' '.join(r['answer'].split())}")
print_table([[c["claim"], "yes" if c["supported"] else "**no**"] for c in r["claims"]],
            ["Claim", "Supported by the context?"])

**A pipeline, m01:** The 2025 Nobel Prize in Physics went to John Clarke, Michel Devoret, and John M. Martinis "for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit" [3][4]. The passages do not say which universities these laureates are affiliated with, so I can't answer that part. Passage [5] gives only a general count of Yale-affiliated Nobel laureates and doesn't name any of them, and [4] mentions Duke University only as the organization behind an earlier 2014 prize.

| Claim                                                                                                                                  | Supported by the context?   |
|:---------------------------------------------------------------------------------------------------------------------------------------|:----------------------------|
| The 2025 Nobel Prize in Physics went to John Clarke, Michel Devoret, and John M. Martinis.                                             | yes                         |
| The prize was awarded "for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit." | yes                         |
| Passage [5] gives only a general count of Yale-affiliated Nobel laureates without naming any of them.                                  | yes                         |
| Passage [4] mentions Duke University only as the organization behind an earlier 2014 prize (Fritz London Memorial Prize).              | yes                         |

### Validate the judge

The book warns: check the judge before trusting it, and use a rule where a rule can decide.
Here correctness has both, so we can compare them answer by answer.

In [12]:
pairs = [(name, qid) for name in ["A pipeline", "B agent"] for qid, *_ in BENCHMARK]
agree = sum(results[n][q]["correct"] == results[n][q]["judge_correct"] for n, q in pairs)
disagree = [[n, q, "correct" if results[n][q]["correct"] else "not correct",
             "correct" if results[n][q]["judge_correct"] else "not correct",
             " ".join(results[n][q]["answer"].split())[:150] + "..."]
            for n, q in pairs if results[n][q]["correct"] != results[n][q]["judge_correct"]]
display_md(f"Rule and judge agree on **{agree} of {len(pairs)}** answers.")
if disagree:
    print_table(disagree, ["System", "Question", "Rule", "Judge", "Answer (start)"])

Rule and judge agree on **48 of 48** answers.

Agreement is high here, because the facts are short names that are easy to judge. Where rule
and judge disagree, read the answer: either the rule is too strict (a fact written in another
form) or the judge is too lenient (an answer that sounds complete but is not). While this
notebook was written, the one disagreement exposed a bug in the rule: it dropped the dash between
"Wisconsin" and "Madison" in a university name, and so no longer found the word "Wisconsin". The check works in both
directions.

### Diagnosing failures

The book's diagnostic matrix puts retrieval and answer quality side by side. We call retrieval
complete when the context holds every expected fact (context recall 1).

In [13]:
for name in ["A pipeline", "B agent"]:
    cells = {(rc, ac): [] for rc in (False, True) for ac in (False, True)}
    for qid, *_ in BENCHMARK:
        r = results[name][qid]
        cells[(r["recall"] == 1.0, r["correct"])].append(qid)
    display_md(f"**{name}**")
    print_table([
        ["Low", "Low", len(cells[(False, False)]), "fix retrieval", ", ".join(cells[(False, False)])],
        ["High", "Low", len(cells[(True, False)]), "fix generation", ", ".join(cells[(True, False)])],
        ["Low", "High", len(cells[(False, True)]), "suspicious: answered from memory?",
         ", ".join(cells[(False, True)])],
        ["High", "High", len(cells[(True, True)]), "working", ", ".join(cells[(True, True)])],
    ], ["Retrieval", "Answer", "Questions", "Diagnosis", "Ids"])

**A pipeline**

| Retrieval   | Answer   |   Questions | Diagnosis                         | Ids                                                                  |
|:------------|:---------|------------:|:----------------------------------|:---------------------------------------------------------------------|
| Low         | Low      |           9 | fix retrieval                     | s08, s09, m01, m02, m03, m05, m06, m07, m08                          |
| High        | Low      |           1 | fix generation                    | s07                                                                  |
| Low         | High     |           0 | suspicious: answered from memory? |                                                                      |
| High        | High     |          14 | working                           | s01, s02, s03, s04, s05, s06, s10, s11, s12, m04, m09, m10, m11, m12 |

**B agent**

| Retrieval   | Answer   |   Questions | Diagnosis                         | Ids                                                                                                               |
|:------------|:---------|------------:|:----------------------------------|:------------------------------------------------------------------------------------------------------------------|
| Low         | Low      |           0 | fix retrieval                     |                                                                                                                   |
| High        | Low      |           1 | fix generation                    | s07                                                                                                               |
| Low         | High     |           0 | suspicious: answered from memory? |                                                                                                                   |
| High        | High     |          23 | working                           | s01, s02, s03, s04, s05, s06, s08, s09, s10, s11, s12, m01, m02, m03, m04, m05, m06, m07, m08, m09, m10, m11, m12 |

Almost all of the pipeline's failures are retrieval failures, the multi-hop questions above
all, and the fix for them is exactly what the agent does. The "suspicious" row stays empty for
both systems: neither answered without evidence, although the model alone shows that it knows
most of the 2025 answers. The grounded prompt held.

Question s07 is the one "fix generation" case, and it is instructive. Both systems answered
honestly that they could not find the 2026 literature laureate. The rule still counts their
retrieval as complete, because a passage about the *2025* prize lists Anne Carson among the
betting favourites. A fact string in a passage is not yet evidence for the fact: rule-based
context recall is a cheap proxy that a judge, or a reader, has to check where it matters.

## Comparing Two Configurations

Is the agent better than the pipeline, or did it get lucky? The book's paired sign test keeps
only the questions where the two systems disagree and asks how likely so uneven a split would be
if both were equally good.

That likelihood is the **p-value**, written $p$. We pretend the two systems are equally good, so
each disagreement goes to either one with the toss of a fair coin. Then $p$ is the probability
that chance alone produces a split at least as one-sided as the one we observed, in either
direction. How to read it:

- **Small $p$** (below the common threshold of 0.05): a split this uneven would rarely happen by
  chance, so we conclude that one system really is better. $p = 0.01$ means one time in a hundred.
- **Large $p$**: chance alone produces such a split often, so the benchmark gives no evidence of a
  difference. $p = 0.5$ means every second time. This does not prove that the systems are equally
  good; it means the benchmark cannot tell them apart.

A $p$-value says how much to trust a difference, not how large it is. It shrinks with the number
of disagreements: the same ratio becomes convincing on a larger benchmark. We check the function
on the book's two examples first.

In [14]:
def sign_test(wins_b, wins_a):
    """Two-sided paired sign test: p-value of a split at least this uneven."""
    n, k = wins_a + wins_b, max(wins_a, wins_b)
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, i) for i in range(k, n + 1)) / 2 ** n
    return min(1.0, 2 * tail)

print_table([["50 questions: B wins 8, A wins 2", f"{sign_test(8, 2):.3f}"],
             ["100 questions: B wins 16, A wins 4", f"{sign_test(16, 4):.3f}"]],
            ["Book example", "p"])

| Book example                       |     p |
|:-----------------------------------|------:|
| 50 questions: B wins 8, A wins 2   | 0.109 |
| 100 questions: B wins 16, A wins 4 | 0.012 |

Both match the book: $p \approx 0.11$ and $p \approx 0.012$. Now our benchmark:

In [15]:
def compare(name_a, name_b):
    a_only = [q for q, *_ in BENCHMARK if results[name_a][q]["correct"] and not results[name_b][q]["correct"]]
    b_only = [q for q, *_ in BENCHMARK if results[name_b][q]["correct"] and not results[name_a][q]["correct"]]
    both = sum(results[name_a][q]["correct"] and results[name_b][q]["correct"] for q, *_ in BENCHMARK)
    return [f"{name_b} vs {name_a}", both, f"{len(b_only)} ({', '.join(b_only) or '-'})",
            f"{len(a_only)} ({', '.join(a_only) or '-'})", f"{sign_test(len(b_only), len(a_only)):.3f}"]

print_table([compare("A pipeline", "B agent"), compare("C model only", "A pipeline")],
            ["Comparison", "Both correct", "Only the second correct", "Only the first correct", "p"])

| Comparison                 |   Both correct | Only the second correct                         | Only the first correct           |     p |
|:---------------------------|---------------:|:------------------------------------------------|:---------------------------------|------:|
| B agent vs A pipeline      |             14 | 9 (s08, s09, m01, m02, m03, m05, m06, m07, m08) | 0 (-)                            | 0.004 |
| A pipeline vs C model only |             10 | 4 (s01, s10, s11, m04)                          | 6 (m01, m02, m03, m06, m07, m08) | 0.754 |

The agent against the pipeline is conclusive even on 24 questions: every disagreement goes the
agent's way, and a split that one-sided rarely happens by chance. The pipeline against the model
alone is the opposite case. Their totals are close and their disagreements split both ways, so
the test finds no evidence that either is better. Yet they are not equivalent: they fail on
*different* questions, the pipeline on multi-hop questions about 2025, the model on 2026. A
single score hides that; the list of disagreements shows it.

Whether the agent's gain is worth its price is the product decision the book describes: about
three times the tokens and several times the latency of the pipeline, fixed best before the
experiment, not after it.

<div style="border-left: 4px solid #C8102E; background: rgba(200, 16, 46, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#C8102E; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Measure retrieval, answer, and price separately, then test the difference</strong><br>
Context recall shows <em>why</em> a system fails, correctness shows <em>whether</em> it fails, and latency and tokens show what each answer costs. A higher score for B is a hypothesis; only the paired test on the questions where A and B disagree tells whether the benchmark supports it.
</div>

## When Not to Use RAG

The model-only system C is the book's first case: knowledge that is stable and already in the
model. Split by the year the question asks about:

In [16]:
rows = []
for name in SYSTEMS:
    by_year = {False: [], True: []}
    for qid, *_ in BENCHMARK:
        by_year[qid in YEAR_2026].append(results[name][qid]["correct"])
    rows.append([name, f"{sum(by_year[False])} / {len(by_year[False])}",
                 f"{sum(by_year[True])} / {len(by_year[True])}",
                 f"{np.mean([results[name][q]['rt'].tokens for q, *_ in BENCHMARK]):,.0f}"])
print_table(rows, ["System", "Correct: up to 2025", "Correct: 2026", "Tokens per answer"])

| System       | Correct: up to 2025   | Correct: 2026   |   Tokens per answer |
|:-------------|:----------------------|:----------------|--------------------:|
| A pipeline   | 11 / 17               | 3 / 7           |               1,930 |
| B agent      | 17 / 17               | 6 / 7           |               5,958 |
| C model only | 16 / 17               | 0 / 7           |                 258 |

For what the model already knows, the model alone is the cheapest system and nearly always
right; the pipeline is even worse there, because a retrieval miss makes its grounded prompt
refuse an answer the model knows. For 2026, past the training data, the model alone answers
nothing, and retrieval is what makes the system work. An agent can make this choice per
question: answer from memory when it is safe, retrieve when the question needs current,
private, or verifiable facts.

## Summary

| Dimension | Measured here | How |
|---|---|---|
| Context recall | expected facts in the retrieved context | rule |
| Context precision | context items holding an expected fact | rule |
| Price | median and p95 seconds, tokens, model and tool calls | recorded per call |
| Correctness, completeness | expected facts in the answer | rule |
| Faithfulness | claims supported by the context | LLM judge, checked against the rule |
| A versus B | questions where the systems disagree | paired sign test |

The chapter's systems differ in a pattern the book predicts: the agent retrieves what multi-hop
questions need and pays for it in tokens and time; the model alone is cheapest and fails exactly
where its training data ends.

## Try it yourself

1. Add ten more questions to `BENCHMARK` (the knowledge base has articles about all 2025
   laureates). Run with AWS access. Does the sign test now separate the pipeline from the model
   alone?
2. Give the pipeline 10 passages instead of 5 (`search(question, 10)` in `system_a`). How do its
   context recall on multi-hop questions, its precision, and its tokens change?
3. Replace the rule-based correctness by the judge's verdict in `compare`. Does the conclusion
   of the sign test change?